<h1>Silver Master Schedule</h1>

<h2>Imports</h2>

<h3>Spark and Initialization</h3>

In [11]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Warsaw_Bus_Project") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

<h3>Imports</h3>

In [ ]:
from datetime import datetime , timedelta
import sys
import gc
sys.path.append('../work')
from config import db_properties,DB_CONFIG,jdbc_url
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import pandas as pd
import numpy as np
import pyspark.pandas as ps
from pyspark.sql.types import FloatType, IntegerType, BooleanType
from pyspark.sql import Row
import psycopg2
from psycopg2 import sql
import pyspark.sql.functions as sf
import zipfile
import urllib.request
from pyspark.testing import assertDataFrameEqual
from pyspark.sql import functions as F
import geopandas as gpd
from shapely.geometry import Point

<h2>Reading Silver GTFS Data</h2>

In [13]:
df_stop_times_silver = spark.read.jdbc(url=jdbc_url, table="silver.stop_times", properties=db_properties)
df_trips_silver = spark.read.jdbc(url=jdbc_url, table="silver.trips", properties=db_properties)
df_routes_silver = spark.read.jdbc(url=jdbc_url, table="silver.routes", properties=db_properties)
df_stops_silver = spark.read.jdbc(url=jdbc_url, table="silver.stops", properties=db_properties)

<h2>Master Schedule</h2>

<h3>Transformations</h3>

In [14]:
df_master_schedule = df_stop_times_silver \
    .join(df_trips_silver,"trip_id","inner") 
df_master_schedule = df_master_schedule \
    .join(df_routes_silver,"route_id","inner") 
df_master_schedule = df_master_schedule \
    .join(df_stops_silver,"stop_id","inner")

In [15]:
df_master_schedule = df_master_schedule\
    .withColumn("schedule_id", F.md5(F.concat_ws("-",F.col("trip_id"),F.col("stop_sequence"),F.col("arrival_time"))))


In [16]:
df_master_schedule = df_master_schedule\
                    .select(F.col("schedule_id"),F.col("trip_id"),F.col("stop_id"),F.col("route_id"),
                            F.col("route_name"),F.col("stop_name"),F.col("district"),
                            F.col("stop_sequence"),F.col("stop_lat"),F.col("stop_lon"),
                            F.col("arrival_time"),F.col("departure_time")                            )
df_master_schedule.show(5)

+--------------------+--------------------+-------+--------+----------+--------------------+--------------+-------------+---------+---------+-------------------+-------------------+
|         schedule_id|             trip_id|stop_id|route_id|route_name|           stop_name|      district|stop_sequence| stop_lat| stop_lon|       arrival_time|     departure_time|
+--------------------+--------------------+-------+--------+----------+--------------------+--------------+-------------+---------+---------+-------------------+-------------------+
|59416e0b01e0948d2...|2026-04-16:102:Pc...| 200104|     102|       102|     Al. Zieleniecka|Praga Południe|            4|52.247387|21.048454|2026-04-16 05:50:00|2026-04-16 05:50:00|
|7204459338d350968...|2026-04-16:102:Pc...| 123204|     102|       102|  Most Świętokrzyski|  Praga Północ|            1|  52.2437|21.039875|2026-04-16 05:46:00|2026-04-16 05:46:00|
|57b73200fa6a93ea2...|2026-04-16:102:Pc...| 210871|     102|       102|PKP Olszynka Groc..

<h3>Writing to DB</h3>

In [17]:
df_master_schedule.write.jdbc(url=jdbc_url, table="silver.master_schedule_staging", mode="overwrite", properties=db_properties)

<h3>Swaping staged table with production</h3>

In [18]:
conn = psycopg2.connect(**DB_CONFIG)
conn.autocommit = False
cur = conn.cursor()

try:
    cur.execute("""
                DROP TABLE IF EXISTS silver.master_schedule_backup;
                ALTER TABLE IF EXISTS silver.master_schedule RENAME TO master_schedule_backup;
                ALTER TABLE IF EXISTS silver.master_schedule_staging RENAME TO master_schedule;""")
    conn.commit()
except Exception as e:
    conn.rollback()
    print(f"Error during table swap: {e}")
finally:
    cur.close()
    conn.close()

In [19]:
spark.catalog.clearCache()
gc.collect()
spark.stop()